# Step 2: Ablation Studies (Text / Image / Tabular)

Step 2에서는 Step 1에서 생성한 `Text + Image + Tabular concat` fusion embedding과 각 single modality embedding을 비교합니다. 목적은 Text, Image, Tabular 각각이 어떤 representation을 만들고, 세 modality를 함께 concat한 fusion 결과가 단일 modality와 얼마나 다른지 확인하는 것입니다.

비교 대상:
- Text-only: MiniLM feature bank -> TextTower -> 64D
- Image-only: CLIP feature bank -> ImageTower -> 64D
- Tabular-only: SVD feature bank -> TabularTower -> 64D
- Text + Image + Tabular concat: Step 1의 `emb_game_concat_64.npy`

생성/갱신 산출물:
- `game_fusion/emb_game_text_only_64.npy`, `.csv`
- `game_fusion/emb_game_image_only_64.npy`, `.csv`
- `game_fusion/emb_game_tabular_only_64.npy`, `.csv`
- `game_fusion/ablation_embedding_distribution.png`


## 1. 라이브러리 및 기본 설정

노트북 실행 위치가 `game_fusion` 내부이든 프로젝트 루트이든 같은 방식으로 동작하도록 `ROOT`를 설정합니다. Plot 저장은 GUI가 없는 환경에서도 동작하도록 `Agg` backend를 사용합니다.

In [ ]:
import importlib.util
import sys
from pathlib import Path
import os
import random

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

ROOT = Path.cwd()
if ROOT.name == "game_fusion":
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 256
SEED = 42


def seed_everything(seed: int = SEED) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


seed_everything(SEED)
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

print(f"Root path: {ROOT}")
print(f"PyTorch device: {DEVICE}")


## 2. Step 1 결과와 Feature Bank 로드

Step 1에서 저장한 `Text + Image + Tabular concat` embedding을 비교 기준으로 로드합니다. Text, Image, Tabular feature bank는 모두 game catalog의 `app_id` 순서에 맞춰 정렬합니다. Image bank는 50,864건이고 game catalog는 50,872건이므로, 이미지가 없는 8건은 `load_image_bank(..., fill_missing=True)` 정책에 따라 평균 vector로 채웁니다.

In [ ]:
from tabular_embedding.tabular_tower import TabularTower, load_tabular_bank

text_tower_path = ROOT / "text_data" / "08_text_tower.py"
spec = importlib.util.spec_from_file_location("text_tower", text_tower_path)
text_tower_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(text_tower_module)
TextTower = text_tower_module.TextTower
load_text_bank = text_tower_module.load_text_bank

image_tower_path = ROOT / "image_embedding" / "07_image_tower.py"
spec = importlib.util.spec_from_file_location("image_tower", image_tower_path)
image_tower_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(image_tower_module)
ImageTower = image_tower_module.ImageTower
load_image_bank = image_tower_module.load_image_bank

games = pd.read_parquet(ROOT / "Data_process" / "games_metadata_enriched.parquet")
game_ids = games["app_id"].to_numpy()

emb_concat_step1 = np.load(ROOT / "game_fusion" / "emb_game_concat_64.npy").astype(np.float32)

text_bank, text_id2row = load_text_bank(
    ROOT / "text_data" / "emb_text_minilm",
    app_ids=game_ids,
    device=DEVICE,
    fill_missing=True,
)

image_bank, image_id2row = load_image_bank(
    ROOT / "image_embedding" / "emb_clip_squash",
    app_ids=game_ids,
    device=DEVICE,
    fill_missing=True,
)

tab_bank, tab_id2row = load_tabular_bank(
    ROOT / "tabular_embedding" / "emb_tabular_svd64",
    app_ids=game_ids,
    device=DEVICE,
    fill_missing=True,
)

print("Loaded Step 1 concat embedding and modality feature banks")
print(f"  Games: {len(game_ids):,}")
print(f"  Step 1 Text+Image+Tabular concat: {emb_concat_step1.shape}")
print(f"  Text bank:    {tuple(text_bank.shape)} {text_bank.dtype}")
print(f"  Image bank:   {tuple(image_bank.shape)} {image_bank.dtype}")
print(f"  Tabular bank: {tuple(tab_bank.shape)} {tab_bank.dtype}")


## 3. Single Modality Embedding 생성 함수

각 modality tower를 단독으로 실행해 64D game embedding bank를 만듭니다. 모든 tower 출력은 L2 normalize되어 있으므로 downstream 비교에서 같은 scale로 다룰 수 있습니다.

In [ ]:
def generate_single_modality_embeddings(tower, bank, batch_size=BATCH_SIZE):
    tower.eval()
    chunks = []
    with torch.no_grad():
        for start_idx in range(0, len(game_ids), batch_size):
            end_idx = min(start_idx + batch_size, len(game_ids))
            emb = tower(bank[start_idx:end_idx])
            chunks.append(emb.cpu().numpy())
    return np.concatenate(chunks, axis=0).astype(np.float32)


def save_embedding_bank(embeddings, stem):
    output_dir = ROOT / "game_fusion"
    npy_path = output_dir / f"{stem}.npy"
    csv_path = output_dir / f"{stem}.csv"
    np.save(npy_path, embeddings)
    pd.DataFrame({"app_id": game_ids}).to_csv(csv_path, index=False)
    print(f"Saved {stem}: {embeddings.shape}")
    return npy_path, csv_path


## 4. Text-only / Image-only / Tabular-only 생성

TextTower, ImageTower, TabularTower를 각각 독립적으로 초기화하고, single modality baseline embedding을 생성합니다. 이 결과는 Step 1의 fusion embedding과 비교하기 위한 ablation 기준입니다.

In [ ]:
text_only_tower = TextTower(in_dim=384, hidden=192, out_dim=64).to(DEVICE)
image_only_tower = ImageTower(in_dim=512, hidden=256, out_dim=64).to(DEVICE)
tab_only_tower = TabularTower(in_dim=64, hidden=128, out_dim=64).to(DEVICE)

print("Generating Text-only embeddings...")
emb_text_only = generate_single_modality_embeddings(text_only_tower, text_bank)

print("Generating Image-only embeddings...")
emb_image_only = generate_single_modality_embeddings(image_only_tower, image_bank)

print("Generating Tabular-only embeddings...")
emb_tab_only = generate_single_modality_embeddings(tab_only_tower, tab_bank)

save_embedding_bank(emb_text_only, "emb_game_text_only_64")
save_embedding_bank(emb_image_only, "emb_game_image_only_64")
save_embedding_bank(emb_tab_only, "emb_game_tabular_only_64")


## 5. Embedding Norm 통계 비교

각 embedding bank의 L2 norm이 의도대로 1 근처인지 확인합니다. Norm이 크게 다르면 concat/fusion 또는 downstream scoring에서 특정 modality가 과도하게 영향을 줄 수 있습니다.

In [12]:
embedding_sets = [
    ("Text-only", emb_text_only),
    ("Image-only", emb_image_only),
    ("Tabular-only", emb_tab_only),
    ("Text+Image+Tabular", emb_concat_step1),
]

summary_rows = []
for name, emb in embedding_sets:
    norms = np.linalg.norm(emb, axis=1)
    summary_rows.append({
        "Model": name,
        "Shape": str(emb.shape),
        "Norm Mean": norms.mean(),
        "Norm Std": norms.std(),
        "Norm Min": norms.min(),
        "Norm Max": norms.max(),
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))


             Model       Shape  Norm Mean     Norm Std  Norm Min  Norm Max
         Text-only (50872, 64)        1.0 3.963720e-08       1.0       1.0
        Image-only (50872, 64)        1.0 3.997405e-08       1.0       1.0
      Tabular-only (50872, 64)        1.0 3.943407e-08       1.0       1.0
Text+Image+Tabular (50872, 64)        1.0 4.083652e-08       1.0       1.0


## 6. 같은 Game 기준 Modality 간 Cosine 비교

동일한 `app_id` 행끼리 cosine similarity를 계산합니다. 이 값은 성능 지표가 아니라 modality representation이 서로 얼마나 다른 방향의 정보를 담는지 보는 sanity check입니다.

In [13]:
def paired_cosine(a, b):
    return np.sum(a * b, axis=1)


pair_rows = [
    ("Text vs Image", paired_cosine(emb_text_only, emb_image_only)),
    ("Text vs Tabular", paired_cosine(emb_text_only, emb_tab_only)),
    ("Image vs Tabular", paired_cosine(emb_image_only, emb_tab_only)),
    ("Text vs Fusion", paired_cosine(emb_text_only, emb_concat_step1)),
    ("Image vs Fusion", paired_cosine(emb_image_only, emb_concat_step1)),
    ("Tabular vs Fusion", paired_cosine(emb_tab_only, emb_concat_step1)),
]

pair_summary = pd.DataFrame([
    {
        "Pair": name,
        "Mean": values.mean(),
        "Std": values.std(),
        "Min": values.min(),
        "Max": values.max(),
    }
    for name, values in pair_rows
])
print(pair_summary.to_string(index=False))


             Pair      Mean      Std       Min      Max
    Text vs Image -0.049827 0.031592 -0.173978 0.080797
  Text vs Tabular  0.016225 0.082589 -0.246515 0.334536
 Image vs Tabular -0.115482 0.070419 -0.411703 0.170775
   Text vs Fusion  0.088361 0.051564 -0.116948 0.273192
  Image vs Fusion -0.057169 0.053497 -0.264203 0.151080
Tabular vs Fusion -0.075917 0.065302 -0.410885 0.204962


## 7. Distribution Plot 저장

Embedding norm 분포와 같은 game 기준 paired cosine similarity 분포를 함께 저장합니다.

**왼쪽 그래프 해석:** 모든 embedding은 `F.normalize`를 거친 L2-normalized vector라서 L2 norm이 거의 1.0입니다. 따라서 Text-only, Image-only, Tabular-only, Fusion의 norm curve가 거의 완전히 겹쳐 하나처럼 보이는 것이 정상입니다. 이를 명확히 보이도록 원래 `L2 norm` 대신 `L2 norm - 1.0`을 그리고, 그래프 안에 overlap annotation을 표시합니다.


In [14]:
# 모든 embedding은 L2-normalized라 norm 분포가 거의 1.0에 겹칩니다.
# 왼쪽 plot은 겹침을 명확히 보여주기 위해 L2 norm 자체가 아니라 (L2 norm - 1.0)을 표시합니다.
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for name, emb in embedding_sets:
    norms = np.linalg.norm(emb, axis=1)
    norm_delta = norms - 1.0
    label = f"{name} (std={norm_delta.std():.1e})"
    sns.kdeplot(norm_delta, label=label, ax=axes[0], linewidth=2)

axes[0].axvline(0.0, color="black", linestyle="--", linewidth=1, alpha=0.7)
axes[0].set_title("L2 Norm Deviation from 1.0 (Curves Overlap)")
axes[0].set_xlabel("L2 norm - 1.0")
axes[0].text(
    0.02,
    0.95,
    "All norm curves overlap\nbecause every embedding is L2-normalized",
    transform=axes[0].transAxes,
    va="top",
    ha="left",
    bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.85},
)
axes[0].legend(fontsize=8)

for name, values in pair_rows:
    sns.kdeplot(values, label=name, ax=axes[1], linewidth=2)

axes[1].set_title("Paired Cosine Similarity")
axes[1].set_xlabel("cosine similarity")
axes[1].legend(fontsize=8)

plt.tight_layout()
fig_path = ROOT / "game_fusion" / "ablation_embedding_distribution.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.close(fig)

print("Norm plot note: curves overlap because all embeddings are L2-normalized.")
print(f"Saved visualization: {fig_path}")


Norm plot note: curves overlap because all embeddings are L2-normalized.
Saved visualization: c:\Users\User\26_2_Contest\game_fusion\ablation_embedding_distribution.png


## 8. Step 2 완료 요약

이 단계의 산출물은 Step 3에서 Step 1 fusion embedding과 single modality embedding을 비교하거나, downstream 추천 실험에서 ablation baseline으로 사용할 수 있습니다.

In [15]:
print("STEP 2 COMPLETE")
print("Generated/updated files:")
print("  emb_game_text_only_64.npy/csv")
print("  emb_game_image_only_64.npy/csv")
print("  emb_game_tabular_only_64.npy/csv")
print("  ablation_embedding_distribution.png")
print("Next: Step 3 BPR-supervised tuning can use the refreshed Step 1/2 outputs.")


STEP 2 COMPLETE
Generated/updated files:
  emb_game_text_only_64.npy/csv
  emb_game_image_only_64.npy/csv
  emb_game_tabular_only_64.npy/csv
  ablation_embedding_distribution.png
Next: Step 3 BPR-supervised tuning can use the refreshed Step 1/2 outputs.
